# 03 · Voice clone — GPT-SoVITS v2ProPlus training

**Kernel:** `JARVIS - GPT-SoVITS` (the package's bundled `runtime\python.exe`)

Pipeline over `JARVIS voice clone.mp3`:
1. **Vocals** — UVR5 separates dialogue from the score (the step that matters most).
2. **Slice + measure** — every segment gets a music-bleed figure, a speaker-similarity score and ASR confidence.
3. **Select** — keep the cleanest 1–3 minutes, then *listen* to what was kept.
4. **Format** — GPT-SoVITS feature extraction (text/BERT, HuBERT, speaker embedding, semantic tokens).
5. **Train SoVITS**, then **6. train GPT** — small batches for 8 GB.

Every GPU step runs in its own process, so this kernel never holds VRAM while training.
Before starting: close the Android emulators and don't run notebooks 01/02/04 at the same time.

**Limits of the automatic filtering.** Speaker similarity assumes JARVIS is the majority voice in the file.
There is no automatic reverb detector here that I would trust. The listening pass in step 3 is the real check.

In [ ]:
import os, sys, json, time, shutil, subprocess, threading
from pathlib import Path
import numpy as np

GSV_ROOT = next(p for p in sorted(Path(r"C:\jarvis-apps").glob("GPT-SoVITS*")) if (p / "webui.py").is_file())
PY = GSV_ROOT / "runtime" / "python.exe"
PRE = "GPT_SoVITS/pretrained_models"                    # relative to GSV_ROOT, as the package expects
SOURCE = Path(r"P:\Coding\App Development\MyProjects\JARVIS\audio\source\JARVIS voice clone.mp3")
WORK = Path(r"C:\jarvis-data\voice")                     # no spaces: GPT-SoVITS shells out with these paths
NB_OUT = Path.cwd() / "outputs"
EXP_NAME, VERSION = "jarvis_v2pp", "v2ProPlus"
OPT_DIR = GSV_ROOT / "logs" / EXP_NAME

TARGET_SECONDS = 150                                     # aim for 1-3 minutes of the cleanest audio
MIN_SEG_S, MAX_SEG_S = 2.0, 12.0
SOVITS_BATCH, SOVITS_EPOCHS, SOVITS_SAVE_EVERY = 4, 8, 4 # drop batch to 2 if SoVITS runs out of memory
GPT_BATCH, GPT_EPOCHS, GPT_SAVE_EVERY = 4, 15, 5

for d in ["00_source", "01_uvr/vocal", "01_uvr/instrumental", "02_slices", "03_selected"]:
    (WORK / d).mkdir(parents=True, exist_ok=True)
NB_OUT.mkdir(exist_ok=True)
assert PY.is_file() and SOURCE.is_file()
print(GSV_ROOT, "|", subprocess.run([str(PY), "--version"], capture_output=True, text=True).stdout.strip())

# The embedded runtime ignores PYTHONPATH (python39._pth) and finds the package's modules through users.pth,
# which ships pointing at the packager's D:\ drive. webui.py rewrites it on every launch; do the same here.
USERS_PTH = GSV_ROOT / "runtime" / "Lib" / "site-packages" / "users.pth"
USERS_PTH.write_text("\n".join(str(GSV_ROOT / p) for p in
                               ["", "GPT_SoVITS/BigVGAN", "tools", "tools/asr", "GPT_SoVITS", "tools/uvr5"]) + "\n")

def gsv_env(**extra):
    env = os.environ.copy()
    env.update({
        "PATH": f"{GSV_ROOT};{GSV_ROOT / 'runtime'};" + env["PATH"],   # bundled ffmpeg/ffprobe
        "TEMP": str(GSV_ROOT / "TEMP"), "version": VERSION, "is_half": "True",
        "PYTHONIOENCODING": "utf-8", "HF_HOME": r"C:\jarvis-models\hf",
    })
    env.update({k: str(v) for k, v in extra.items()})
    return env

def gpu_used_mb():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return int(out.strip().splitlines()[0])

WHISPER_PY = Path(r"C:\jarvis-venvs\whisper\Scripts\python.exe")  # transcription uses the known-good whisper venv

def run_gsv(args, label, python=None, **env_extra):
    """Run a step in its own process (GPT-SoVITS runtime unless `python` is given), stream its output,
    record wall time and peak GPU use."""
    (GSV_ROOT / "TEMP").mkdir(exist_ok=True)
    before, peak, done = gpu_used_mb(), [0], threading.Event()
    def poll():
        while not done.wait(1.0):
            peak[0] = max(peak[0], gpu_used_mb())
    threading.Thread(target=poll, daemon=True).start()
    t0 = time.perf_counter()
    if python is None:
        cmd, cwd, env = [str(PY), "-s", *map(str, args)], GSV_ROOT, gsv_env(**env_extra)
    else:                                     # a different venv: don't leak GPT-SoVITS paths into it
        env = os.environ.copy()
        env.update({"PYTHONIOENCODING": "utf-8", "HF_HOME": r"C:\jarvis-models\hf"})
        env.update({k: str(v) for k, v in env_extra.items()})
        cmd, cwd = [str(python), *map(str, args)], WORK
    proc = subprocess.Popen(cmd, cwd=cwd, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    done.set()
    result = {"step": label, "exit": rc, "minutes": round((time.perf_counter() - t0) / 60, 1),
              "gpu_used_before_mb": before, "gpu_used_peak_mb": max(peak[0], before)}
    print(f"\n[{label}] {result}")
    if rc != 0:
        raise RuntimeError(f"{label} failed (exit {rc}) - see output above")
    return result

run_log = []

In [ ]:
# Training wants the whole GPU. List every process holding dedicated memory on the NVIDIA adapter.
import ctypes, psutil
from ctypes import wintypes

class _FmtValue(ctypes.Structure):
    _fields_ = [("CStatus", wintypes.DWORD), ("largeValue", ctypes.c_longlong)]
class _FmtItem(ctypes.Structure):
    _fields_ = [("szName", wintypes.LPWSTR), ("FmtValue", _FmtValue)]

def pdh_read(path):
    pdh, q, c = ctypes.WinDLL("pdh"), wintypes.HANDLE(), wintypes.HANDLE()
    pdh.PdhOpenQueryW(None, None, ctypes.byref(q))
    pdh.PdhAddEnglishCounterW(q, path, None, ctypes.byref(c))
    pdh.PdhCollectQueryData(q)
    size, count = wintypes.DWORD(0), wintypes.DWORD(0)
    pdh.PdhGetFormattedCounterArrayW(c, 0x400, ctypes.byref(size), ctypes.byref(count), None)
    buf = (ctypes.c_byte * size.value)()
    pdh.PdhGetFormattedCounterArrayW(c, 0x400, ctypes.byref(size), ctypes.byref(count), buf)
    pdh.PdhCloseQuery(q)
    items = ctypes.cast(buf, ctypes.POINTER(_FmtItem * count.value)).contents
    return {i.szName: i.FmtValue.largeValue for i in items if i.szName}

adapters = pdh_read(r"\GPU Adapter Memory(*)\Dedicated Usage")
nvidia_luid = max(adapters, key=adapters.get).split("_phys")[0]   # the iGPU has no dedicated memory
hogs = []
for name, used in pdh_read(r"\GPU Process Memory(*)\Dedicated Usage").items():
    if nvidia_luid in name and used > 100 * 2**20:
        pid = int(name.split("_")[1])
        try:
            proc_name = psutil.Process(pid).name()
        except psutil.Error:
            proc_name = "?"
        hogs.append((round(used / 2**20), pid, proc_name))
print(f"nvidia-smi used now: {gpu_used_mb()} MB")
for mb, pid, proc_name in sorted(hogs, reverse=True):
    print(f"  {mb:6d} MB  pid {pid:<6} {proc_name}")
if any("qemu" in n or "python" in n for _, _, n in hogs):
    print("\nWARNING: an emulator or another Python process holds GPU memory. Close it before training.")

## Step 1 · Separate vocals from the score (UVR5)

In [ ]:
SRC_WAV = WORK / "00_source" / "jarvis_source.wav"
subprocess.run([str(GSV_ROOT / "runtime" / "ffmpeg.exe"), "-y", "-hide_banner", "-loglevel", "error", "-i", str(SOURCE),
                "-vn", "-ac", "2", "-ar", "44100", "-acodec", "pcm_s16le", str(SRC_WAV)], check=True)

weights_dir = GSV_ROOT / "tools" / "uvr5" / "uvr5_weights"
available = sorted(p.stem for p in weights_dir.iterdir() if p.suffix in {".pth", ".ckpt"})
print("UVR5 models in the package:", available)
PREFERENCE = ["bs_roformer", "mel_band_roformer", "HP2", "HP3", "HP5"]   # best dialogue-vs-score separation first
UVR_MODEL = next((m for pref in PREFERENCE for m in available if pref.lower() in m.lower()), None)
assert UVR_MODEL, "no usable UVR5 vocal model found"
print("using:", UVR_MODEL)

UVR_CODE = r"""
import os
from bsroformer import Roformer_Loader
from vr import AudioPre
name, root = os.environ["uvr_model"], "tools/uvr5/uvr5_weights"
if "roformer" in name.lower():
    m = Roformer_Loader(model_path=f"{root}/{name}.ckpt", config_path=f"{root}/{name}.yaml", device="cuda", is_half=True)
else:
    m = AudioPre(agg=10, model_path=f"{root}/{name}.pth", device="cuda", is_half=True)
m._path_audio_(os.environ["uvr_input"], os.environ["uvr_ins"], os.environ["uvr_vocal"], "wav", "HP3" in name)
print("separation finished")
"""
REDO_UVR = False   # separation is slow; reuse earlier stems unless asked
stems_exist = all(any((WORK / "01_uvr" / d).glob("*.wav")) for d in ["vocal", "instrumental"])
if REDO_UVR or not stems_exist:
    for d in ["vocal", "instrumental"]:
        for f in (WORK / "01_uvr" / d).glob("*"):
            f.unlink()
    run_log.append(run_gsv(["-c", UVR_CODE], f"UVR5 {UVR_MODEL}", uvr_model=UVR_MODEL, uvr_input=SRC_WAV,
                           uvr_vocal=WORK / "01_uvr" / "vocal", uvr_ins=WORK / "01_uvr" / "instrumental"))
else:
    print("reusing existing UVR5 stems (set REDO_UVR = True to separate again)")
VOCAL_WAV = max((WORK / "01_uvr" / "vocal").glob("*.wav"), key=os.path.getmtime)
INST_WAV = max((WORK / "01_uvr" / "instrumental").glob("*.wav"), key=os.path.getmtime)
print("vocal:", VOCAL_WAV.name, "| instrumental:", INST_WAV.name)

In [ ]:
# Listen to 20 s of each stem before going further: if the vocal stem still carries obvious score, stop here.
from IPython.display import Audio, display
import librosa
for label, path in [("vocal stem", VOCAL_WAV), ("instrumental stem", INST_WAV)]:
    y, sr = librosa.load(str(path), sr=22050, mono=True, offset=60, duration=20)
    print(label); display(Audio(y, rate=sr))

## Step 2 · Slice and measure every segment

In [ ]:
sys.path.insert(0, str(GSV_ROOT / "tools"))
from slicer2 import Slicer
import soundfile as sf

vocal, _ = librosa.load(str(VOCAL_WAV), sr=32000, mono=True)
inst, _ = librosa.load(str(INST_WAV), sr=32000, mono=True)
n = min(len(vocal), len(inst)); vocal, inst = vocal[:n], inst[:n]

def rms_db(x):
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

slicer = Slicer(sr=32000, threshold=-34, min_length=4000, min_interval=300, hop_size=10, max_sil_kept=500)  # webui defaults
SEG_DIR = WORK / "02_slices"
for f in SEG_DIR.glob("*.wav"):
    f.unlink()
rows = []
for item in slicer.slice(vocal):
    if len(item) != 3:
        continue
    chunk, start, end = item
    peak = float(np.abs(chunk).max())
    if peak < 1e-3 or (end - start) < 0.5 * 32000:
        continue
    c = chunk / peak if peak > 1 else chunk.copy()
    c = c / np.abs(c).max() * (0.9 * 0.25) + 0.75 * c                        # same normalisation as slice_audio.py
    name = f"seg_{start:010d}_{end:010d}.wav"
    sf.write(SEG_DIR / name, (c * 32767).astype(np.int16), 32000)
    rows.append({"name": name, "start_s": round(start / 32000, 1), "dur_s": round((end - start) / 32000, 2),
                 "bleed_db": round(rms_db(inst[start:end]) - rms_db(chunk), 1),   # score level relative to the voice
                 "clip_frac": round(float(np.mean(np.abs(chunk) > 0.99)), 4)})

durs = np.array([r["dur_s"] for r in rows])
in_range = (durs >= MIN_SEG_S) & (durs <= MAX_SEG_S)
print(f"{len(rows)} segments, {durs.sum():.0f} s total; {in_range.sum()} in {MIN_SEG_S}-{MAX_SEG_S} s ({durs[in_range].sum():.0f} s)")

In [ ]:
# Two GPU passes, each in its own process:
#  1. speaker embeddings with the package's ERes2NetV2 (GPT-SoVITS runtime)
#  2. transcripts + confidence with faster-whisper large-v3-turbo in the whisper venv. The package's own
#     faster-whisper 1.1.1 / ctranslate2 3.24 pairing is mismatched and it ships no English ASR model.
SV_JSON, ASR_JSON = WORK / "segment_speaker.json", WORK / "segment_asr.json"
SV_CODE = r"""
import os, json, glob
import torch, librosa
from sv import SV
sv, out = SV("cuda", True), {}
for f in sorted(glob.glob(os.path.join(os.environ["seg_dir"], "*.wav"))):
    wav, _ = librosa.load(f, sr=16000)
    e = sv.compute_embedding3(torch.from_numpy(wav).unsqueeze(0).cuda()).float()
    out[os.path.basename(f)] = torch.nn.functional.normalize(e, dim=-1)[0].cpu().numpy().tolist()
json.dump(out, open(os.environ["out_path"], "w"))
print("speaker embeddings for", len(out), "segments")
"""
ASR_CODE = r"""
import os, json, glob, importlib.util
from pathlib import Path
import sys
dirs = [Path(sys.prefix) / "cudnn9"]                # cuDNN 9 copied from the torch wheel; nvidia wheels if present
for pkg in ("nvidia.cublas", "nvidia.cudnn"):
    spec = importlib.util.find_spec(pkg)
    if spec and spec.submodule_search_locations:
        dirs.append(Path(list(spec.submodule_search_locations)[0]) / "bin")
for d in [d for d in dirs if d.is_dir()]:
    os.add_dll_directory(str(d))
    os.environ["PATH"] = str(d) + os.pathsep + os.environ["PATH"]
import numpy as np
from faster_whisper import WhisperModel
model, out = WhisperModel("large-v3-turbo", device="cuda", compute_type="int8_float16"), {}
for f in sorted(glob.glob(os.path.join(os.environ["seg_dir"], "*.wav"))):
    segs = list(model.transcribe(f, language="en", beam_size=5, vad_filter=False, condition_on_previous_text=False)[0])
    out[os.path.basename(f)] = {
        "text": " ".join(s.text.strip() for s in segs).strip(),
        "avg_logprob": float(np.mean([s.avg_logprob for s in segs])) if segs else -9.0,
        "no_speech_prob": float(max([s.no_speech_prob for s in segs], default=1.0)),
        "compression_ratio": float(max([s.compression_ratio for s in segs], default=0.0))}
json.dump(out, open(os.environ["out_path"], "w", encoding="utf-8"), ensure_ascii=False)
print("transcribed", len(out), "segments")
"""
ASR_MODEL = "large-v3-turbo int8_float16 (whisper venv)"
run_log.append(run_gsv(["-c", SV_CODE], "speaker embeddings", seg_dir=SEG_DIR, out_path=SV_JSON))
run_log.append(run_gsv(["-c", ASR_CODE], "ASR labels", python=WHISPER_PY, seg_dir=SEG_DIR, out_path=ASR_JSON))
speaker = json.loads(SV_JSON.read_text())
asr = json.loads(ASR_JSON.read_text(encoding="utf-8"))
metrics = {name: {**asr[name], "embedding": speaker[name]} for name in asr}
for r in rows:
    r.update({k: v for k, v in metrics[r["name"]].items() if k != "embedding"})

## Step 3 · Select the cleanest 1–3 minutes

In [ ]:
# Cosine similarity to the dominant speaker (ERes2NetV2). Measured on this file: every segment scores 0.66-0.92 with
# no second cluster, i.e. no sign of other characters. The 0.66-0.70 tail is the 99-130 s stretch where the score is
# loudest, so 0.70 acts as a second guard against music-corrupted segments rather than a speaker filter.
SPK_SIM_MIN = 0.70
# Score level relative to the voice in the ORIGINAL mix during the segment (from the instrumental stem).
# Measured on this file: <= -15 dB keeps only 42 s, <= -10 dB 152 s, <= -6 dB 202 s, <= -3 dB 274 s.
# So -6 is the hard cut, and the ranking below fills the target from the least-scored segments first.
BLEED_DB_MAX = -6.0
LOGPROB_MIN = -0.60     # low ASR confidence usually means overlap, mumbling or artefacts
NO_SPEECH_MAX = 0.50
MANUAL_REJECT = set()   # after listening below, add bad segment names here and re-run this cell

E = np.array([metrics[r["name"]]["embedding"] for r in rows])
speechy = np.array([(MIN_SEG_S <= r["dur_s"] <= MAX_SEG_S) and r["no_speech_prob"] <= NO_SPEECH_MAX for r in rows])
S = E[speechy] @ E[speechy].T
medoid = S.mean(1).argmax()
core = np.argsort(-S[medoid])[: max(3, len(S) // 2)]                 # the half most similar to the medoid
centroid = E[speechy][core].mean(0); centroid /= np.linalg.norm(centroid)
for r, e in zip(rows, E):
    r["spk_sim"] = round(float(e @ centroid), 3)
    reasons = []
    if not MIN_SEG_S <= r["dur_s"] <= MAX_SEG_S: reasons.append("length")
    if r["no_speech_prob"] > NO_SPEECH_MAX or len(r["text"].split()) < 2: reasons.append("no speech")
    if r["spk_sim"] < SPK_SIM_MIN: reasons.append("other speaker")
    if r["bleed_db"] > BLEED_DB_MAX: reasons.append("music bleed")
    if r["avg_logprob"] < LOGPROB_MIN or r["compression_ratio"] > 2.4: reasons.append("low ASR confidence")
    if r["clip_frac"] > 0.001: reasons.append("clipping")
    if r["name"] in MANUAL_REJECT: reasons.append("manual")
    r["reject"] = reasons
    # Least score under the voice dominates (-15 dB -> +0.45, -6 dB -> +0.18); speaker match and ASR confidence break ties.
    r["score"] = r["spk_sim"] - 0.03 * r["bleed_db"] + 0.1 * r["avg_logprob"]

passing = sorted([r for r in rows if not r["reject"]], key=lambda r: -r["score"])
selected, total = [], 0.0
for r in passing:
    if total >= TARGET_SECONDS:
        break
    selected.append(r); total += r["dur_s"]

from collections import Counter
print("rejections:", dict(Counter(reason for r in rows for reason in r["reject"])))
print(f"passing {len(passing)} segments ({sum(r['dur_s'] for r in passing):.0f} s) -> selected {len(selected)} ({total:.0f} s)")
if total < 60:
    print("WARNING: under 1 minute survived. Quality will suffer - loosen thresholds only after listening to why.")

print("\nLowest speaker similarity among segments with speech (check these are really other characters):")
for r in sorted([r for r, ok in zip(rows, speechy) if ok], key=lambda r: r["spk_sim"])[:8]:
    print(f"  {r['spk_sim']:.2f}  {r['name']}  {r['text'][:80]}")

SEL_DIR, LIST_PATH = WORK / "03_selected", WORK / "jarvis.list"
for f in SEL_DIR.glob("*.wav"):
    f.unlink()
lines = []
for r in selected:
    shutil.copy2(SEG_DIR / r["name"], SEL_DIR / r["name"])
    lines.append(f"{SEL_DIR / r['name']}|jarvis|EN|{r['text']}")
LIST_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")

ref_candidates = [r for r in selected if 3.2 <= r["dur_s"] <= 9.5]     # GPT-SoVITS needs a 3-10 s reference
ref = max(ref_candidates, key=lambda r: r["score"])
(WORK / "ref.json").write_text(json.dumps({"path": str(SEL_DIR / ref["name"]), "text": ref["text"]}, indent=2), encoding="utf-8")
(WORK / "selection.json").write_text(json.dumps([{k: v for k, v in r.items()} for r in rows], indent=2), encoding="utf-8")
print(f"\nreference clip: {ref['name']} ({ref['dur_s']} s) -> {ref['text']!r}")

In [ ]:
# Listen. This is the real quality gate: music under the voice, a second voice, or a roomy/echoey sound
# -> add the name to MANUAL_REJECT in the cell above and re-run it.
from IPython.display import Markdown
REVIEW_N = 30
for r in selected[:REVIEW_N]:
    display(Markdown(f"`{r['name']}` · {r['dur_s']} s · speaker {r['spk_sim']:.2f} · bleed {r['bleed_db']} dB — {r['text']}"))
    display(Audio(str(SEL_DIR / r["name"])))

## Step 4 · Format the dataset (text/BERT, HuBERT, speaker embedding, semantic tokens)

In [ ]:
RESET = True    # wipe previous features for this experiment; stale features silently survive otherwise
if RESET and OPT_DIR.exists():
    shutil.rmtree(OPT_DIR)
OPT_DIR.mkdir(parents=True, exist_ok=True)
common = dict(inp_text=LIST_PATH, inp_wav_dir=SEL_DIR, exp_name=EXP_NAME, opt_dir=OPT_DIR,
              i_part=0, all_parts=1, _CUDA_VISIBLE_DEVICES=0)

run_log.append(run_gsv(["GPT_SoVITS/prepare_datasets/1-get-text.py"], "1a text + BERT",
                       bert_pretrained_dir=f"{PRE}/chinese-roberta-wwm-ext-large", **common))
part = OPT_DIR / "2-name2text-0.txt"
(OPT_DIR / "2-name2text.txt").write_text(part.read_text(encoding="utf8").strip("\n") + "\n", encoding="utf8"); part.unlink()

ssl = dict(cnhubert_base_dir=f"{PRE}/chinese-hubert-base", sv_path=f"{PRE}/sv/pretrained_eres2netv2w24s4ep4.ckpt")
run_log.append(run_gsv(["GPT_SoVITS/prepare_datasets/2-get-hubert-wav32k.py"], "1b HuBERT + wav32k", **ssl, **common))
run_log.append(run_gsv(["GPT_SoVITS/prepare_datasets/2-get-sv.py"], "1b speaker embeddings", **ssl, **common))
run_log.append(run_gsv(["GPT_SoVITS/prepare_datasets/3-get-semantic.py"], "1c semantic tokens",
                       pretrained_s2G=f"{PRE}/v2Pro/s2Gv2ProPlus.pth", s2config_path="GPT_SoVITS/configs/s2v2ProPlus.json", **common))
part = OPT_DIR / "6-name2semantic-0.tsv"
(OPT_DIR / "6-name2semantic.tsv").write_text("item_name\tsemantic_audio\n" + part.read_text(encoding="utf8").strip("\n") + "\n", encoding="utf8"); part.unlink()

counts = {"list": len(lines),
          "2-name2text": len((OPT_DIR / "2-name2text.txt").read_text(encoding="utf8").strip().splitlines()),
          "4-cnhubert": len(list((OPT_DIR / "4-cnhubert").glob("*.pt"))),
          "7-sv_cn": len(list((OPT_DIR / "7-sv_cn").glob("*.pt"))),
          "6-name2semantic": len((OPT_DIR / "6-name2semantic.tsv").read_text(encoding="utf8").strip().splitlines()) - 1}
print(counts)
assert len(set(counts.values())) == 1, "a formatting step dropped segments - check the output above"

## Step 5 · Train SoVITS

In [ ]:
s2 = json.loads((GSV_ROOT / "GPT_SoVITS/configs/s2v2ProPlus.json").read_text(encoding="utf8"))
s2["train"].update(batch_size=SOVITS_BATCH, epochs=SOVITS_EPOCHS, text_low_lr_rate=0.4,
                   pretrained_s2G=f"{PRE}/v2Pro/s2Gv2ProPlus.pth", pretrained_s2D=f"{PRE}/v2Pro/s2Dv2ProPlus.pth",
                   if_save_latest=True, if_save_every_weights=True, save_every_epoch=SOVITS_SAVE_EVERY,
                   gpu_numbers="0", grad_ckpt=False, lora_rank="32")
s2["model"]["version"] = VERSION
s2["data"]["exp_dir"] = s2["s2_ckpt_dir"] = OPT_DIR.as_posix()
s2.update(save_weight_dir=f"SoVITS_weights_{VERSION}", name=EXP_NAME, version=VERSION)
(OPT_DIR / f"logs_s2_{VERSION}").mkdir(parents=True, exist_ok=True)
S2_CFG = GSV_ROOT / "TEMP" / "tmp_s2_jarvis.json"
S2_CFG.parent.mkdir(exist_ok=True)
S2_CFG.write_text(json.dumps(s2), encoding="utf8")

run_log.append(run_gsv(["GPT_SoVITS/s2_train.py", "--config", S2_CFG], "SoVITS training"))
sovits_weights = sorted((GSV_ROOT / f"SoVITS_weights_{VERSION}").glob(f"{EXP_NAME}_e*.pth"), key=os.path.getmtime)
print("SoVITS weights:", [p.name for p in sovits_weights])

## Step 6 · Train GPT

In [ ]:
import yaml
s1 = yaml.safe_load((GSV_ROOT / "GPT_SoVITS/configs/s1longer-v2.yaml").read_text(encoding="utf8"))
s1["train"].update(batch_size=GPT_BATCH, epochs=GPT_EPOCHS, save_every_n_epoch=GPT_SAVE_EVERY,
                   if_save_every_weights=True, if_save_latest=True, if_dpo=False,
                   half_weights_save_dir=f"GPT_weights_{VERSION}", exp_name=EXP_NAME)
s1.update(pretrained_s1=f"{PRE}/s1v3.ckpt",
          train_semantic_path=(OPT_DIR / "6-name2semantic.tsv").as_posix(),
          train_phoneme_path=(OPT_DIR / "2-name2text.txt").as_posix(),
          output_dir=(OPT_DIR / f"logs_s1_{VERSION}").as_posix())
(OPT_DIR / "logs_s1").mkdir(exist_ok=True)
S1_CFG = GSV_ROOT / "TEMP" / "tmp_s1_jarvis.yaml"
S1_CFG.write_text(yaml.dump(s1, default_flow_style=False), encoding="utf8")

run_log.append(run_gsv(["GPT_SoVITS/s1_train.py", "--config_file", S1_CFG], "GPT training", _CUDA_VISIBLE_DEVICES=0, hz="25hz"))
gpt_weights = sorted((GSV_ROOT / f"GPT_weights_{VERSION}").glob(f"{EXP_NAME}-e*.ckpt"), key=os.path.getmtime)
print("GPT weights:", [p.name for p in gpt_weights])

In [ ]:
summary = {"uvr_model": UVR_MODEL, "asr_model": ASR_MODEL, "segments_total": len(rows),
           "segments_selected": len(selected), "seconds_selected": round(total, 1),
           "thresholds": {"spk_sim_min": SPK_SIM_MIN, "bleed_db_max": BLEED_DB_MAX, "logprob_min": LOGPROB_MIN},
           "manual_rejects": sorted(MANUAL_REJECT), "reference": json.loads((WORK / "ref.json").read_text()),
           "sovits": {"batch": SOVITS_BATCH, "epochs": SOVITS_EPOCHS, "weights": [p.name for p in sovits_weights]},
           "gpt": {"batch": GPT_BATCH, "epochs": GPT_EPOCHS, "weights": [p.name for p in gpt_weights]},
           "steps": run_log}
(NB_OUT / "gptsovits_train_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
for s in run_log:
    print(f"{s['step']:28s} {s['minutes']:6.1f} min   GPU peak {s['gpu_used_peak_mb']} MB")